In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║ vast_qlora_train.py — QLoRA RAFT Qwen3.5-4B di Vast.ai (RTX A5000 24GB)     ║
# ║                                                                            ║
# ║ Versi untuk Vast.ai (bukan Colab). A5000 = Ampere -> bf16 native -> cepat   ║
# ║ & stabil (tanpa masalah GradScaler T4). 24GB -> tanpa OOM.                  ║
# ║                                                                            ║
# ║ Cara pakai: jalankan sel berurutan di Jupyter. Upload `raft_train.jsonl`    ║
# ║ ke folder yang sama dulu (lihat README / instruksi chat).                   ║
# ╚══════════════════════════════════════════════════════════════════════════╝


# RAFT QLoRA — Qwen3.5-4B @ Vast.ai (RTX A5000 24GB)
Beda dari versi Colab: **bf16 native** (Ampere), batch lebih besar, download via Jupyter.
Validitas tetap: base = `Qwen/Qwen3.5-4B`, `enable_thinking=False`, export GGUF Q4_K_M.


In [ ]:
# ── 0. Cek GPU ────────────────────────────────────────────────────────────────
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"  # anti-fragmentasi OOM
import subprocess, torch
print(subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total',
                      '--format=csv,noheader'], capture_output=True, text=True).stdout)
assert torch.cuda.is_available(), 'GPU tidak terdeteksi'
print('CUDA:', torch.cuda.get_device_name(0),
      '| bf16 native:', torch.cuda.is_bf16_supported())
# Ampere: aktifkan TF32 -> matmul lebih cepat tanpa kehilangan akurasi berarti
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True


In [ ]:
# ── 1. Install deps ───────────────────────────────────────────────────────────
# Template Vast sudah ada PyTorch; tinggal tambah library training.
# Pakai sys.executable agar install PASTI ke Python kernel yang sama (hindari
# ModuleNotFoundError). Tunggu sampai muncul "Successfully installed ...".
import sys
get_ipython().system(f'{sys.executable} -m pip install -U "transformers>=4.46" "peft>=0.13" "trl>=0.12" "bitsandbytes>=0.44" "accelerate>=1.0" "datasets>=3.0" sentencepiece')


In [ ]:
# ── 1b. Kernel SSM fast-path (percepat training di Ampere) ────────────────────
# Qwen3.5 hybrid pakai SSM; kernel ini mengaktifkan "fast path" -> jauh lebih cepat.
# Dibungkus try/except: kalau gagal kompilasi, OTOMATIS dilewati (training tetap jalan).
import sys, subprocess
try:
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'causal-conv1d', 'flash-linear-attention'],
                   check=True)
    print('✅ Kernel SSM fast-path terpasang')
except Exception as e:
    print('⚠️ Kernel SSM gagal dipasang -> dilewati (training tetap jalan, agak lebih lambat):', e)


In [ ]:
# ── 2. Parameter (disetel untuk A5000 24GB) ───────────────────────────────────
HF_MODEL_ID   = "Qwen/Qwen3.5-4B"     # dikonfirmasi via feasibility check
DATA_PATH     = "raft_train.jsonl"     # upload ke folder ini dulu
OUT_LORA      = "raft_lora"
OUT_MERGED    = "raft_merged"
OUT_GGUF      = "qwen35-4b-raft-q4_k_m.gguf"

MAX_SEQ_LEN   = 2048
LORA_R        = 16
LORA_ALPHA    = 32
LORA_DROPOUT  = 0.05
EPOCHS        = 3
LR            = 2e-4
BATCH         = 2          # vocab 248k bikin logits besar -> batch 2 aman di 24GB
GRAD_ACCUM    = 8          # effective batch = BATCH*GRAD_ACCUM = 16 (sama)
WARMUP_RATIO  = 0.03
VAL_FRACTION  = 0.08
SEED          = 42


In [ ]:
# ── 3. Verifikasi base model ──────────────────────────────────────────────────
from huggingface_hub import model_info
mi = model_info(HF_MODEL_ID)
print('OK:', mi.id)


In [ ]:
# ── 4. Load model 4-bit (QLoRA, compute bf16) ─────────────────────────────────
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
bnb = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,    # Ampere -> bf16 native
)
tokenizer = AutoTokenizer.from_pretrained(HF_MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(
    HF_MODEL_ID, quantization_config=bnb, device_map="auto",
    dtype=torch.bfloat16, trust_remote_code=True)
model.config.use_cache = False
print('Model loaded (4-bit, compute bf16)')


In [ ]:
# ── 5. LoRA (target attention + FFN + SSM) ────────────────────────────────────
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
model = get_peft_model(model, LoraConfig(
    r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT, bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj",
                    "in_proj_qkv", "in_proj_a", "in_proj_b", "in_proj_z", "out_proj"]))
model.print_trainable_parameters()


In [ ]:
# ── 6. Dataset (chat template, NON-THINKING) ──────────────────────────────────
import json
from datasets import Dataset
rows = [json.loads(l) for l in open(DATA_PATH, encoding='utf-8')]
print('Total triple:', len(rows))
def to_text(rec):
    try:
        return tokenizer.apply_chat_template(rec["messages"], tokenize=False,
                                             add_generation_prompt=False, enable_thinking=False)
    except TypeError:
        return tokenizer.apply_chat_template(rec["messages"], tokenize=False,
                                             add_generation_prompt=False)
texts = [{"text": to_text(r)} for r in rows]
ds = Dataset.from_list(texts).train_test_split(test_size=VAL_FRACTION, seed=SEED)
print(ds)
print(ds['train'][0]['text'][:500])


In [ ]:
# ── 7. Collator ───────────────────────────────────────────────────────────────
# DataCollatorForCompletionOnlyLM dihapus di TRL baru -> full-sequence SFT (collator default).
collator = None


In [ ]:
# ── 8. Training (bf16) ────────────────────────────────────────────────────────
from trl import SFTTrainer, SFTConfig
cfg = SFTConfig(
    output_dir=OUT_LORA, per_device_train_batch_size=BATCH,
    gradient_accumulation_steps=GRAD_ACCUM, num_train_epochs=EPOCHS,
    learning_rate=LR, warmup_ratio=WARMUP_RATIO, lr_scheduler_type="cosine",
    logging_steps=10, eval_strategy="epoch",      # eval 1x/epoch -> overhead minimal
    per_device_eval_batch_size=1,                 # vocab 248k -> eval batch 1 (hindari OOM)
    save_strategy="epoch",                        # checkpoint per epoch (disk cukup, aman)
    bf16=True, fp16=False, optim="paged_adamw_8bit", max_length=MAX_SEQ_LEN,
    gradient_checkpointing=True, report_to="none", seed=SEED,
    dataset_text_field="text", packing=False,
)
trainer = SFTTrainer(model=model, args=cfg, train_dataset=ds['train'],
                     eval_dataset=ds['test'], data_collator=collator)
trainer.train()
trainer.save_model(OUT_LORA)
print('✓ LoRA tersimpan di', OUT_LORA)


In [ ]:
# ── 9. Merge LoRA -> bf16 ─────────────────────────────────────────────────────
from peft import AutoPeftModelForCausalLM
del model, trainer; torch.cuda.empty_cache()
merged = AutoPeftModelForCausalLM.from_pretrained(OUT_LORA, dtype=torch.bfloat16, device_map="cpu")
merged = merged.merge_and_unload()
merged.save_pretrained(OUT_MERGED, safe_serialization=True)
tokenizer.save_pretrained(OUT_MERGED)
print('✓ Model merged ->', OUT_MERGED)


In [ ]:
# ── 10. Konversi GGUF Q4_K_M (llama.cpp) ──────────────────────────────────────
get_ipython().system('git clone --depth 1 https://github.com/ggerganov/llama.cpp 2>/dev/null')
get_ipython().system('pip install -r llama.cpp/requirements.txt')
get_ipython().system(f'python llama.cpp/convert_hf_to_gguf.py {OUT_MERGED} --outfile raft_f16.gguf --outtype f16')
get_ipython().system('cd llama.cpp && cmake -B build -DLLAMA_CURL=OFF >/dev/null 2>&1 && cmake --build build --config Release -j --target llama-quantize >/dev/null 2>&1')
get_ipython().system(f'./llama.cpp/build/bin/llama-quantize raft_f16.gguf {OUT_GGUF} Q4_K_M')
print('✓ GGUF siap:', OUT_GGUF)


In [ ]:
# ── 11. Download GGUF ──────────────────────────────────────────────────────────
import os
print('File hasil ada di:', os.path.abspath(OUT_GGUF))
print('Ukuran:', round(os.path.getsize(OUT_GGUF)/1e9, 2), 'GB')
print('\nCARA DOWNLOAD ke laptop:')
print('  Di Jupyter (panel file kiri) -> klik kanan file', OUT_GGUF, '-> Download.')
print('  Setelah turun, pindahkan ke laptop lalu lanjut Fase 3 (daftar ke Ollama).')
